# **Procesamiento de Lenguaje Natural**
## **MNA - Tecnológico de Monterrey**
## **Algunos ejemplos para extraer el texto de un archivo de audio.**

## 🗺️ Mapa del Notebook — ¿Qué hace cada parte?

Este notebook compara **cuatro enfoques para transcribir audio a texto (ASR — Automatic Speech Recognition)** usando el mismo archivo de audio (~73 seg), para que puedas comparar calidad, costo y requerimientos directamente.

| Caso | Modelo | Proveedor | API Key | Cómputo local | Formato entrada |
|------|--------|-----------|---------|----------------|-----------------|
| **1** | whisper-large-v3-turbo | OpenAI open-source vía HuggingFace | No | Sí (GPU recomendada) | MP3 / WAV / M4A |
| **2** | whisper-1 | OpenAI API (pago) | Sí | No (cloud) | MP3 / WAV / M4A |
| **3** | wav2vec2-large-xlsr-53 | Meta open-source vía HuggingFace | No (HF login) | Sí (GPU recomendada) | MP3 re-muestreado a 16 kHz |
| **4** | Google Speech | Google (gratuito con límites de rate) | No | No (cloud) | **WAV obligatorio** |

**Flujo lógico:**
```
Audio MP3 (~73 seg)
    ├── Caso 1: librosa + HuggingFace pipeline   → texto + timestamps por chunk
    ├── Caso 2: OpenAI API (whisper-1)            → Transcription object con puntuación
    ├── Caso 3: Wav2Vec2 local (re-muestreo 16kHz)→ texto sin puntuación ni mayúsculas
    └── Caso 4: SpeechRecognition + Google        → texto con puntuación básica (requiere WAV)
```

---

## 🔑 Conceptos clave

| Concepto | Qué es | Para qué sirve aquí |
|----------|--------|---------------------|
| **ASR** | Convertir audio a texto automáticamente | El objetivo de todo el notebook |
| **Whisper** | Transformer encoder-decoder de OpenAI entrenado con 680k horas de audio | Base de Casos 1 y 2 |
| **Wav2Vec2** | Modelo de Meta con representaciones auto-supervisadas de audio crudo | Caso 3 — arquitectura distinta a Whisper |
| **CTC** | Mecanismo para alinear frames de audio con caracteres sin segmentación previa | Decodificación en Wav2Vec2 |
| **Sample rate** | Frecuencia de muestreo del audio en Hz (ej. 16,000 Hz = 16 kHz) | Wav2Vec2 exige exactamente 16,000 Hz |
| **`return_timestamps=True`** | Divide audio largo en ventanas de 30 seg para procesar con Whisper | Obligatorio para audios >30 segundos |
| **Pipeline HuggingFace** | Abstracción que combina modelo + procesador + inferencia en una sola llamada | Simplifica el Caso 1 |

---

## 💼 Casos de uso de negocio

| Industria | Problema real | Caso recomendado | Por qué |
|-----------|---------------|-----------------|---------|
| **Medios / Streaming** | Subtítulos automáticos para videos | Caso 1 | Timestamps + open-source + local |
| **Call Centers / CX** | Transcribir llamadas para análisis | Caso 2 | Alta calidad, escalable vía API |
| **Legal / Compliance** | Audiencias o juntas a texto buscable | Caso 1 | Local — datos no salen de tu infraestructura |
| **Educación** | Transcribir clases grabadas para notas | Caso 4 | Gratuito para bajo volumen |
| **Producto Edge / offline** | App sin acceso a internet | Caso 3 | Completamente local, sin dependencias cloud |
| **Dictado médico** | Notas de voz a expediente clínico | Caso 1 o 2 | Calidad y fiabilidad críticas |

---

## ⚠️ Advertencias técnicas

**Sesgos presentes:**
- Whisper fue entrenado mayoritariamente con audio en inglés — el español funciona bien pero acentos regionales muy marcados reducen la precisión.
- Wav2Vec2 no produce puntuación ni mayúsculas — el texto de salida requiere post-procesamiento (ej. `deepmultilingualpunctuation`) para uso real.
- El modelo `wav2vec2-large-xlsr-53-spanish` fue ajustado con datos de ciertos dialectos; errores como "poso" por "pozo" son esperables en otros acentos.

**¿Cumplimos el objetivo?**
- ✅ Los 4 casos transcriben correctamente el audio de prueba (~73 seg, español).
- ✅ Caso 2 produce la mejor puntuación y formato gracias al procesamiento en servidores OpenAI.
- ⚠️ Caso 3 introduce errores menores ("poso" en lugar de "pozo") por sesgo dialectal del modelo.
- ⚠️ Caso 1 genera un warning sobre `forced_decoder_ids` al especificar `language="es"` — es inofensivo.

**Escalabilidad a producción:**
- Audios largos con Whisper: `return_timestamps=True` activa chunking automático de 30 seg (ya incluido en el Caso 1).
- Para audio en streaming (tiempo real): usar `WhisperForConditionalGeneration` directamente con buffers de audio.
- Wav2Vec2 con audios largos: dividir en ventanas de ≤30 seg manualmente antes de inferir — no tiene chunking automático.
- Datos sensibles (médico, legal): usar Caso 1 o 3 — el audio no se envía a ningún servidor externo.

# **Caso - 1: modelos whisper de OpenAI con HuggingFace**

## 🏷️ Variantes de Whisper — ¿cuál elegir?

Todas las variantes de la familia Whisper son de código abierto (licencia MIT) y están disponibles en HuggingFace sin necesidad de API key.

| Modelo | Parámetros | RAM aprox. | Velocidad relativa | Cuándo usarlo |
|--------|-----------|------------|-------------------|---------------|
| `whisper-tiny` | 39 M | ~1 GB | ⚡⚡⚡ Muy rápido | Prototipado rápido, bajo cómputo |
| `whisper-base` | 74 M | ~1 GB | ⚡⚡⚡ Rápido | Experimentos iniciales |
| `whisper-small` | 244 M | ~2 GB | ⚡⚡ Moderado | Balance para demos |
| `whisper-medium` | 769 M | ~5 GB | ⚡ Lento en CPU | Producción con GPU disponible |
| `whisper-large-v3` | 1,550 M | ~10 GB | 🐢 Muy lento en CPU | Máxima calidad, GPU obligatoria |
| **`whisper-large-v3-turbo`** ✅ | **809 M** | **~6 GB** | **⚡⚡ Moderado** | **Mejor balance calidad/velocidad** |

**¿Por qué `v3-turbo` como opción por defecto?**
`whisper-large-v3-turbo` es una versión destilada de `large-v3` que mantiene ~95% de la calidad con la mitad de los parámetros del decoder. Es la recomendación actual de OpenAI para casos de uso generales.

* #### **Se tienen diferentes variantes de código abierto de whisper de OpenAI con HuggingFace:**

 https://huggingface.co/models?sort=trending&search=openai%2Fwhisper-

* #### **Estos modelos no requieren el uso de la API de OpenAI.**

## 📦 Librerías del Caso 1

| Librería | Para qué sirve en este notebook |
|----------|--------------------------------|
| `transformers` | Framework de HuggingFace para descargar y usar modelos pre-entrenados (Whisper, Wav2Vec2...) |
| `accelerate` | Optimiza el uso de hardware: selecciona GPU/CPU automáticamente y paraleliza en múltiples GPUs |
| `ffmpeg` | Decodifica formatos de audio/video (MP3, MP4, etc.) — `librosa` lo necesita para leer archivos que no son WAV nativos |
| `librosa` | Análisis de audio: carga archivos, convierte sample rates, extrae características de la señal |

> **Nota:** `ffmpeg` es una dependencia del sistema, no de Python. En tu computadora se instala con el gestor de paquetes del sistema (Homebrew en Mac, apt en Linux); en Colab el comando `apt-get` lo instala directamente.

#### **Se recomienda el uso de GPU**

In [1]:
!pip install transformers accelerate librosa gdown
# En Mac/Linux instala ffmpeg con: brew install ffmpeg  (o apt-get install ffmpeg en Linux)
# Si ya tienes ffmpeg instalado (ej. via faster-whisper) puedes omitir esa línea.

In [2]:
import torch
from transformers import pipeline
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import librosa   # paquete para audio y música


import warnings
# para filtrar advertencias:
warnings.filterwarnings("ignore")

/Users/joelbecerril/miniconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🎧 El archivo de audio de prueba

Se descarga el archivo `20029-01.mp3` del **Proyecto Gutenberg** — una grabación de LibriVox de las *Fábulas de Esopo* narrada en español por un voluntario.

| Propiedad | Valor |
|-----------|-------|
| Duración | ~73 segundos |
| Idioma | Español |
| Formato | MP3 |
| Fuente | Dominio público (Project Gutenberg) |
| Contenido | Fábula #31: "Las ranas y el pantano seco" |

Este archivo es útil porque:
- Tiene duración suficiente para probar el chunking automático de Whisper (>30 seg).
- El narrador tiene un acento neutro — buen caso base para comparar la precisión entre modelos.
- Es de dominio público, sin restricciones de uso.

**¿Por qué `gdown`?** El archivo está en Google Drive en la versión original del notebook (Colab). Localmente podrías descargarlo con `wget` o `requests`.

In [3]:
# Descargamos el archivo de audio:
!gdown "https://www.gutenberg.org/files/20029/mp3/20029-01.mp3"

Downloading...
From: https://www.gutenberg.org/files/20029/mp3/20029-01.mp3
To: /Users/joelbecerril/MNA_WORKSPACE/NLP/Sem7_1jun/Tarea_act4/20029-01.mp3
100%|██████████████████████████████████████| 1.18M/1.18M [00:00<00:00, 1.89MB/s]


In [4]:
# Con librosa podemos obtener información del archivo:
audio_path = "./20029-01.mp3"
audio, sample_rate = librosa.load(audio_path)
print(f"Duración del audio: {len(audio)/sample_rate:.1f} segundos")

Duración del audio: 73.2 segundos


In [5]:
# <Opcional: si deseas escucharlo>
from IPython.display import Audio
Audio(audio_path)

## ⚙️ GPU vs CPU — impacto en ASR

Whisper es un modelo grande (809M–1,550M parámetros). El dispositivo disponible afecta directamente el tiempo de inferencia:

| Hardware | Tiempo aprox. para 73 seg de audio | Cuándo aplica |
|----------|-------------------------------------|---------------|
| CPU (8 cores) | ~4–8 minutos con `large-v3-turbo` | Laptop sin GPU / Mac Intel |
| GPU T4 (Colab free) | ~30–60 segundos | Google Colab gratuito |
| GPU A100 (Colab Pro) | ~5–10 segundos | Colab Pro |

**Regla práctica:** Para audio de hasta ~5 minutos, una GPU T4 es suficiente. Para procesar muchas horas de audio en lote, escalar a A100 o usar la API del Caso 2.

El código `"cuda" if torch.cuda.is_available() else "cpu"` asigna automáticamente el mejor dispositivo disponible — no requiere cambio manual.

In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"   # GPU integrado de Apple Silicon (M1/M2/M3/M4/M5)
else:
    device = "cpu"

print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


## 🎛️ Parámetros ajustables — Selección y carga del modelo

| Parámetro / Elección | Valor actual | Cuándo cambiarlo |
|---------------------|-------------|-----------------|
| `model_id` | `whisper-large-v3-turbo` | Cámbialo a `whisper-small` si tienes <2 GB de RAM o necesitas velocidad. Usa `large-v3` para máxima calidad |
| `torch_dtype` | `torch.float32` | Cambia a `torch.float16` si tienes GPU para reducir memoria a la mitad con mínima pérdida de calidad |
| `.to(device)` | GPU si disponible | Automático — no requiere cambio manual |

**¿Cómo funciona la descarga?**
`AutoProcessor.from_pretrained(model_id)` y `AutoModelForSpeechSeq2Seq.from_pretrained(model_id)` descargan el modelo de HuggingFace Hub la primera vez y lo guardan en caché (`~/.cache/huggingface/`). Las ejecuciones siguientes cargan desde caché sin volver a descargar.

📥 **Datos de entrada:** `model_id` (string con el nombre del modelo) → descarga el par procesador + modelo.
📤 **Salida:** `processor` (tokenizador + extractor de features) y `modelo` (pesos del modelo en memoria).

In [7]:
# Puedes seleccionar algunas de las siguientes versiones de whisper.
# Revisa la documentación de cada uno en HuggingFace.

#model_id="openai/whisper-tiny"    # el más sencillo y ligero, pero menos preciso.
#model_id="openai/whisper-small"
#model_id="openai/whisper-base"
#model="openai/whisper-medium"  # Generalmente el modelo intermedio en cuanto a desempeño y tamaño.
#model_id = "openai/whisper-large-v3"    # mejor modelo
model_id = "openai/whisper-large-v3-turbo"    # una versión ligera de v3

# Cargamos el modelo:
processor = AutoProcessor.from_pretrained(model_id)
modelo = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device)


Loading weights: 100%|██████████| 587/587 [00:00<00:00, 24859.21it/s]


## 🔧 Pipeline de HuggingFace — ¿qué es y qué hace?

`pipeline("automatic-speech-recognition", ...)` es una **abstracción de alto nivel** que encapsula:
1. Convertir el audio crudo a mel-spectrograma (entrada del encoder).
2. Tokenizar las predicciones del decoder.
3. Ejecutar la inferencia del modelo encoder-decoder.
4. Decodificar los tokens predichos a texto.

Sin el pipeline tendrías que hacer esos 4 pasos manualmente (como se hace en el Caso 3 con Wav2Vec2).

| Parámetro del pipeline | Valor | Qué configura |
|-----------------------|-------|---------------|
| `model` | `modelo` (objeto cargado) | El modelo Whisper que usará para inferir |
| `tokenizer` | `processor.tokenizer` | Convierte tokens numéricos ↔ texto |
| `feature_extractor` | `processor.feature_extractor` | Convierte audio crudo → mel-spectrograma |
| `torch_dtype` | `torch.float32` | Precisión numérica de los cálculos |
| `device` | `"cuda"` o `"cpu"` | Hardware donde corre la inferencia |

> **Por qué se pasan `tokenizer` y `feature_extractor` por separado:** el objeto `processor` de Whisper combina ambos, pero el pipeline los necesita de forma individual para poder manejar batches de audio.

In [8]:
# Inicializamos el pipeline:
pipe = pipeline("automatic-speech-recognition",
                model=modelo,
                tokenizer=processor.tokenizer,
                feature_extractor=processor.feature_extractor,
                torch_dtype=torch.float32,      #torch.float16 if torch.cuda.is_available() else torch.float32,
                device=device
                )

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


## 🎛️ Parámetros de la inferencia — `pipe()`

| Parámetro | Valor | Por qué es necesario |
|-----------|-------|----------------------|
| `audio` | array numpy del audio | El audio ya cargado por librosa |
| `return_timestamps=True` | True | **Obligatorio para audios >30 seg.** Whisper procesa ventanas de 30 seg; este flag activa el ensamblado automático de los chunks con sus marcas de tiempo |
| `generate_kwargs={"language":"es"}` | `"es"` | Fuerza la transcripción en español. Sin esto, Whisper detecta el idioma automáticamente (funciona bien, pero es más lento) |

⚠️ **Warning `forced_decoder_ids`:** Al especificar `language="es"`, Whisper puede emitir un warning sobre conflicto con `forced_decoder_ids`. Es un bug conocido de versiones antiguas de `transformers` — no afecta el resultado. La transcripción se genera correctamente en español.

📥 **Datos de entrada:** array numpy `audio` de librosa + parámetros de generación.
📤 **Salida:** diccionario con `"text"` (transcripción completa) y `"chunks"` (lista de segmentos con timestamps).

In [9]:
# Extraemos el texto del audio:
result = pipe(audio,
              return_timestamps=True,   # Para audios mayores a 30 segs.
              generate_kwargs={"language":"es"}  # Lo detecta automático, pero se le puede indicar el idioma.
              )

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

In [10]:
result["text"]   # el texto del audio

' Las Fábulas de Esopo Grabado para LibriVox.org por Paulino Fábula número 31 Las ranas y el pantano seco Vivían dos ranas en un bello pantano, Pero llegó el verano y se secó, por lo cual lo abandonaron para buscar otro con agua. Aliaron en su camino un profundo pozo repleto de agua Y al verlo, dijo una rana a la otra Amiga, bajemos las dos a este pozo Pero, ¿y también se secara el agua de este pozo? Repuso la compañera ¿Cómo crees que subiremos entonces? Al tratar de emprender una acción, analiza primero las consecuencias de ella. Fin de la fábula Esta es una grabación del dominio público. Gracias por ver el video.'

## 📊 Estructura completa del resultado — `result["chunks"]`

El diccionario de salida tiene dos claves:

```python
result = {
    "text":   "transcripción completa como un solo string",
    "chunks": [
        {"timestamp": (0.0, 2.84), "text": " Las fábulas de Esopo"},
        {"timestamp": (2.84, 10.12), "text": " Grabado para LibriVox.org por Paulino"},
        ...
    ]
}
```

| Campo | Tipo | Cómo usarlo |
|-------|------|-------------|
| `result["text"]` | `str` | Transcripción completa para análisis de texto (NLP posterior) |
| `result["chunks"]` | `list[dict]` | Lista de segmentos para generación de subtítulos (SRT/VTT) |
| `chunk["timestamp"]` | `tuple(float, float)` | Tiempo de inicio y fin en segundos dentro del audio original |
| `chunk["text"]` | `str` | Texto del segmento correspondiente a esa ventana |

**Caso de uso práctico — generar subtítulos:**
```python
for chunk in result["chunks"]:
    start, end = chunk["timestamp"]
    print(f"[{start:.1f}s → {end:.1f}s] {chunk['text'].strip()}")
```

Observa que algunos timestamps en la salida real muestran `(0.0, ...)` repetidos — es un artefacto del chunking de Whisper en audios donde los chunks se reinician al pasar de una ventana de 30 seg a la siguiente. En producción se normalizan con un offset acumulativo.

In [11]:
result

{'text': ' Las Fábulas de Esopo Grabado para LibriVox.org por Paulino Fábula número 31 Las ranas y el pantano seco Vivían dos ranas en un bello pantano, Pero llegó el verano y se secó, por lo cual lo abandonaron para buscar otro con agua. Aliaron en su camino un profundo pozo repleto de agua Y al verlo, dijo una rana a la otra Amiga, bajemos las dos a este pozo Pero, ¿y también se secara el agua de este pozo? Repuso la compañera ¿Cómo crees que subiremos entonces? Al tratar de emprender una acción, analiza primero las consecuencias de ella. Fin de la fábula Esta es una grabación del dominio público. Gracias por ver el video.',
 'chunks': [{'timestamp': (0.0, 2.84), 'text': ' Las Fábulas de Esopo'},
  {'timestamp': (2.84, 10.12),
   'text': ' Grabado para LibriVox.org por Paulino'},
  {'timestamp': (10.12, 14.44), 'text': ' Fábula número 31'},
  {'timestamp': (14.44, 20.0), 'text': ' Las ranas y el pantano seco'},
  {'timestamp': (20.0, 26.16),
   'text': ' Vivían dos ranas en un bello 

## 🔄 Caso 2 vs Caso 1 — misma familia, ejecución opuesta

Ambos casos usan modelos Whisper de OpenAI, pero la arquitectura de uso es completamente diferente:

```
Caso 1 (HuggingFace):                     Caso 2 (API OpenAI):
┌────────────────────────┐                ┌──────────────────────────┐
│ Modelo descargado      │                │ Tu máquina               │
│ en tu máquina          │                │   ↓ archivo de audio     │
│ Tu hardware hace la    │                │   ↓ HTTPS a OpenAI       │
│ inferencia             │                │ Servidores OpenAI hacen  │
│ Sin costo por uso      │                │ la inferencia            │
│ Privacidad garantizada │                │ Pago por minuto de audio │
└────────────────────────┘                └──────────────────────────┘
```

| Criterio | Caso 1 | Caso 2 |
|----------|--------|--------|
| Costo | Gratis (pagas el cómputo) | ~$0.006 / minuto de audio |
| Privacidad | Total — datos no salen | Datos enviados a OpenAI |
| Calidad | Alta (`large-v3-turbo`) | Alta (`whisper-1` ≈ `medium`) |
| Configuración | Requiere descargar modelo | Solo necesitas API key |
| Velocidad (1 archivo) | Depende de tu hardware | Rápido (cloud dedicado) |
| Escalabilidad | Limitada por tu GPU | Alta (paralelo en cloud) |

**Cuándo usar Caso 2:** cuando tienes alto volumen de audio, no quieres mantener infraestructura de GPU, o necesitas integración rápida vía API.

# **Caso - 2: modelo whisper-1 de OpenAI ... requiere uso de la API de OpenAI**

> ⚠️ **OMITIDO — Motivo: costo de API**
>
> Este caso requiere una clave de API de OpenAI con saldo activo. El costo es ~$0.006 USD por minuto de audio (para este archivo ~73 seg ≈ $0.0007 USD — casi nada individual, pero se cobra por uso).
>
> **¿Perdemos algo al omitirlo?** No en términos de comprensión del algoritmo:
> - Caso 1 y Caso 2 usan el mismo modelo Whisper de OpenAI — la arquitectura es idéntica.
> - La diferencia es **dónde corre el cómputo**: local (Caso 1) vs. servidores OpenAI (Caso 2).
> - El output del Caso 2 tiene mejor puntuación (los servidores aplican post-procesamiento adicional), pero la transcripción es esencialmente la misma.
>
> **Lo que sí es exclusivo del Caso 2:**
> - El patrón de integración con una API de pago (`openai.audio.transcriptions.create()`).
> - Ver el objeto `Transcription` de respuesta vs el `dict` de HuggingFace.
> - La tabla comparativa de salidas en la celda de más abajo captura estas diferencias sin necesidad de ejecutar.
>
> Para ejecutarlo en el futuro: `export OPENAI_API_KEY="sk-..."` y descomentar las celdas de código.

#### **Revisa la documentación del modelo "whisper-1":** https://platform.openai.com/docs/models/whisper-1

In [12]:
# OMITIDO — descomentar si tienes OPENAI_API_KEY configurada
# !pip install openai

In [13]:
# OMITIDO — descomentar si tienes OPENAI_API_KEY configurada
# import os
# from openai import OpenAI
# import openai

In [14]:
# OMITIDO — descomentar si tienes OPENAI_API_KEY configurada
# api_key = os.environ.get("OPENAI_API_KEY", "")
# if not api_key:
#     raise ValueError("Define la variable de entorno OPENAI_API_KEY o pega tu clave directamente")

## 🎛️ Parámetros de la API — `client.audio.transcriptions.create()`

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `model` | `"whisper-1"` | Único modelo disponible en la API para ASR. Equivale aproximadamente a `whisper-medium` en calidad |
| `file` | `audio_file` (binario) | El archivo de audio abierto en modo `"rb"`. Formatos soportados: mp3, mp4, mpeg, mpga, m4a, wav, webm |
| `language` | `"es"` | Código de idioma ISO 639-1. Opcional — si se omite, Whisper detecta el idioma automáticamente |
| `response_format` | (no especificado → `"json"`) | Otros valores: `"text"`, `"srt"`, `"verbose_json"`, `"vtt"` — útiles para subtítulos directamente |
| `temperature` | (no especificado → `0`) | Controla la aleatoriedad. `0` = determinístico (máxima consistencia) |

📥 **Datos de entrada:** archivo de audio como objeto binario (abrir con `open(..., "rb")`) + configuración de la llamada.
📤 **Salida:** objeto `Transcription` con atributo `.text` (el texto transcrito).

**Límites de la API de OpenAI:**
- Tamaño máximo de archivo: **25 MB**
- Para archivos más grandes: dividir con `pydub` o `ffmpeg` antes de enviar.

In [15]:
# OMITIDO — descomentar si tienes OPENAI_API_KEY configurada
# client = OpenAI(api_key=api_key)
# with open(audio_path, "rb") as audio_file:
#     transcript = client.audio.transcriptions.create(model="whisper-1",
#                                                     file=audio_file,
#                                                     language="es"
#                                                     )

## 📋 Comparación de formatos de salida — Caso 1 vs Caso 2

Ambos modelos transcriben el mismo audio, pero el formato de salida difiere:

**Caso 1 (HuggingFace pipeline):**
```
' Las fábulas de Esopo Grabado para LibriVox.org por Paulino Fábula número 31 ...'
```
→ String con espacio inicial, sin puntuación en separaciones de chunks.

**Caso 2 (API OpenAI):**
```
'Las fábulas de Esopo. Grabado para LibriVox.org por Paulino. Fábula número 31. ...'
```
→ Con puntuación entre oraciones, sin espacio inicial, mejor capitalización.

| Aspecto | Caso 1 | Caso 2 |
|---------|--------|--------|
| Puntuación | Mínima | Buena (puntos, comas, signos de interrogación) |
| Capitalización | Correcta | Más consistente |
| Formato objeto | `dict` con `"text"` y `"chunks"` | Objeto `Transcription` con `.text` |
| Acceso al texto | `result["text"]` | `transcript.text` |
| Timestamps disponibles | Sí (`result["chunks"]`) | Solo con `response_format="verbose_json"` |

La diferencia de puntuación se debe a que el modelo en los servidores de OpenAI aplica post-procesamiento adicional sobre la decodificación de Whisper.

In [16]:
# OMITIDO — descomentar si ejecutaste las celdas anteriores del Caso 2
# transcript.text   # en particular observa el formato de la salida del texto.

## 🔬 Caso 3 — Arquitectura diferente a Whisper

Mientras Whisper es un **encoder-decoder** que genera texto token a token (como un modelo de lenguaje), **Wav2Vec2** funciona de forma muy diferente:

```
Whisper (Casos 1 y 2):
  Audio → Mel-spectrograma → Encoder → Decoder (genera texto autoregresivamente)

Wav2Vec2 (Caso 3):
  Audio crudo → Representación continua → Proyección lineal → CTC → Texto
```

| Característica | Whisper | Wav2Vec2 |
|---------------|---------|----------|
| Arquitectura | Encoder-Decoder (seq2seq) | Encoder + cabeza CTC |
| Entrenamiento | Supervisado con pares audio-texto | Auto-supervisado + fine-tuning |
| Salida | Texto con puntuación y capitalización | Solo letras y espacios (sin puntuación) |
| Idiomas en un modelo | ~100 (multilingüe) | Específico por modelo fine-tuneado |
| Sample rate requerido | Cualquiera (librosa re-muestrea) | **16,000 Hz obligatorio** |
| Chunking automático largo | Sí (`return_timestamps=True`) | No — requiere división manual |

**¿Por qué usar Wav2Vec2 entonces?**
- Es completamente local y open-source (Meta, licencia libre).
- Existe una gran variedad de modelos fine-tuneados por idioma/dialecto específico en HuggingFace.
- Para tareas de clasificación de audio o speech más allá de la transcripción, las representaciones internas de Wav2Vec2 son muy útiles como features.

# **Caso - 3: otros modelos en HuggingFace**

* #### **Los modelos Wav2Vec2 abarcan una gran variedad de problemáticas, son modelos abiertos de Meta y muchos de ellos los podemos encontrar en HuggingFace.**

* #### **Estos modelos fueron entrenados con datos muestreados a 16 Hz, por lo que se debe verificar que los datos de entrada tengan esta característica.**

In [17]:
# transformers y librosa ya instalados en la celda de Caso 1
# Si ejecutas este caso de forma independiente: pip install transformers librosa

In [18]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import librosa
import torch

In [19]:
import os
from huggingface_hub import login

In [20]:
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    raise ValueError(
        "Define la variable de entorno HF_TOKEN antes de correr este notebook.\n"
        "En terminal (antes de abrir VS Code / Jupyter):\n"
        "  export HF_TOKEN='hf_...'\n"
        "O pega el token directamente en esta celda:\n"
        "  hf_token = 'hf_...'"
    )
login(hf_token)

ValueError: Define la variable de entorno HF_TOKEN antes de correr este notebook.
En terminal (antes de abrir VS Code / Jupyter):
  export HF_TOKEN='hf_...'
O pega el token directamente en esta celda:
  hf_token = 'hf_...'

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"   # GPU integrado de Apple Silicon (M1/M2/M3/M4/M5)
else:
    device = "cpu"

print(f"Usando dispositivo: {device}")

## 🔑 Por qué `sr=16000` es obligatorio en Wav2Vec2

Los modelos de la familia Wav2Vec2 fueron pre-entrenados con audio muestreado a **exactamente 16,000 Hz**. Si el audio tiene otro sample rate, el modelo no falla con error — simplemente produce transcripciones incorrectas o sin sentido.

```
Audio original MP3:  sample_rate = 22,050 Hz  (estándar MP3)
                         ↓  librosa.load(sr=16000)
Audio re-muestreado: sample_rate = 16,000 Hz  ← lo que espera el modelo
```

**¿Qué significa 16,000 Hz?**
- El audio original tiene 22,050 muestras por segundo.
- Al re-muestrear a 16,000 Hz, librosa reduce las muestras a 16,000 por segundo mediante interpolación.
- La duración del audio permanece igual (73.2 seg), pero ahora tiene 16,000 × 73.2 = 1,171,200 muestras en lugar de 22,050 × 73.2 = 1,614,060.

**Comparación:**
| Paso | `librosa.load()` en Caso 1 | `librosa.load(sr=16000)` en Caso 3 |
|------|---------------------------|-------------------------------------|
| Sample rate resultado | Nativo del archivo (22,050 Hz) | Forzado a 16,000 Hz |
| Número de muestras | Mayor | Menor |
| Para usar con | Whisper (acepta cualquier SR) | Wav2Vec2 (exige 16,000 Hz) |

In [ ]:
audio_path = "./20029-01.mp3"
audio, sample_rate = librosa.load(audio_path,
                                  sr=16000   # los modelos Wav2Vec2 requieren un muestreo (sample rate) de 16Hz.
                                  )
print(f"Duración del audio: {len(audio)/sample_rate:.1f} segundos")

Duración del audio: 73.2 segundos


## 🎛️ Modelos Wav2Vec2 disponibles para español

| Modelo | Quién lo entrenó | Notas |
|--------|-----------------|-------|
| `facebook/wav2vec2-large-xlsr-53-spanish` ✅ | Meta + comunidad | Modelo por defecto — fine-tuned en Common Voice español |
| `jonatasgrosman/wav2vec2-large-xlsr-53-spanish` | Jonatas Grosman | Fine-tuning adicional, puede dar mejor calidad |
| `facebook/wav2vec2-base-10k-voxpopuli-ft-es` | Meta | Más ligero, entrenado con datos del Parlamento Europeo |

**¿Cómo funciona la descarga?**
Igual que en el Caso 1: `Wav2Vec2Processor.from_pretrained(model_name)` y `Wav2Vec2ForCTC.from_pretrained(model_name)` descargan y cachean el modelo en `~/.cache/huggingface/`.

**Diferencia entre `Wav2Vec2Processor` y `AutoProcessor`:**
- `AutoProcessor` (Caso 1): clase genérica que infiere el procesador correcto para cualquier modelo.
- `Wav2Vec2Processor` (Caso 3): clase específica para Wav2Vec2 — incluye el `feature_extractor` y el `tokenizer` ya combinados.

📥 **Datos de entrada:** `model_name` (string) + `device` (cuda/cpu).
📤 **Salida:** `processor` (preprocesador de audio) y `modelo` (Wav2Vec2ForCTC con cabeza CTC).

In [ ]:
# Modelos entrenados específicamente con datos en español.
# Recuerda revisar la documentación de cada uno en HuggingFace.

model_name = "facebook/wav2vec2-large-xlsr-53-spanish"
#model_name = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"
#model_name="facebook/wav2vec2-base-10k-voxpopuli-ft-es"

# Cargamos el modelo y el procesador:
processor = Wav2Vec2Processor.from_pretrained(model_name)
modelo = Wav2Vec2ForCTC.from_pretrained(model_name).to(device)


## 🔄 Conversión del audio a tensor de PyTorch

`processor(audio, sampling_rate=sample_rate, return_tensors="pt")` realiza dos cosas:
1. **Normaliza** la señal de audio (media cero, varianza unitaria) — requisito de Wav2Vec2.
2. **Empaqueta** el array numpy en un tensor de PyTorch con la forma `(1, n_muestras)`.

```
audio (numpy array):     shape = (1,171,200,)   ← float32
                ↓  processor
input_values:           shape = (1, 1,171,200)  ← tensor PyTorch [batch, muestras]
                ↓  .to(device)
                         GPU / CPU según disponibilidad
```

**¿Por qué `.input_values`?** El processor devuelve un objeto con múltiples campos posibles; `.input_values` es la clave que contiene la señal normalizada lista para el encoder de Wav2Vec2.

📥 **Datos de entrada:** `audio` (array numpy a 16 kHz) + `sample_rate` (16,000).
📤 **Salida:** `input_values` — tensor `(1, n_muestras)` en el dispositivo seleccionado.

In [ ]:
# Convertimos los datos a un tensor de pyTorch con el muestro seleccionado:
input_values = processor(audio,
                         sampling_rate=sample_rate,
                         return_tensors="pt").input_values.to(device)


## ⚡ `torch.no_grad()` — por qué es necesario en inferencia

Durante el **entrenamiento**, PyTorch construye un grafo computacional que almacena todas las operaciones para calcular los gradientes (backpropagation). Esto consume memoria adicional proporcional al tamaño del modelo.

Durante la **inferencia** (predicción) no se necesitan gradientes — solo el resultado final:

```python
# Sin torch.no_grad() → PyTorch guarda el grafo completo en memoria
logits = modelo(input_values).logits    # memoria = pesos + grafo

# Con torch.no_grad() → solo los pesos, sin grafo
with torch.no_grad():
    logits = modelo(input_values).logits  # ~30-50% menos memoria
```

**Regla:** siempre usa `torch.no_grad()` cuando hagas inferencia (no fine-tuning). Es el equivalente de `model.eval()` pero más eficiente en memoria.

📥 **Datos de entrada:** `input_values` — tensor `(1, n_muestras)`.
📤 **Salida:** `logits` — tensor `(1, n_frames, vocab_size)` con las puntuaciones brutas por carácter en cada frame de tiempo.

In [ ]:
# Realizamos las inferencias sin calcular los gradientes para optimizar recursos,
# ya que no estamos haciendo ajuste de parámetros (fine-tuning):
with torch.no_grad():
    logits = modelo(input_values).logits


## 🔤 Decodificación CTC — de logits a texto

La decodificación tiene dos pasos:

**Paso 1 — `torch.argmax(logits, dim=-1)`:** selecciona el carácter de mayor probabilidad en cada frame de tiempo.
```
logits:         shape (1, n_frames, vocab_size)
                         ↓  argmax sobre vocab_size
predicted_ids:  shape (1, n_frames)  ← ID del carácter más probable en cada frame
```

**Paso 2 — `processor.decode(predicted_ids[0])`:** convierte los IDs a texto aplicando las reglas CTC:
- Elimina los tokens `<blank>` (relleno entre caracteres).
- Colapsa caracteres repetidos consecutivos (`aaa` → `a`).
- Convierte IDs numéricos a caracteres del vocabulario.

```
predicted_ids:    [12, 12, 0, 12, 15, 15, 4, ...]   ← IDs con blanks y repeticiones
                       ↓  processor.decode()
transcription:    "las ranas y el pantano..."         ← texto limpio sin puntuación
```

⚠️ **Sin puntuación:** CTC predice carácter a carácter sin contexto de oración — la puntuación no es parte de su vocabulario estándar. Para agregarla en producción se usa un modelo de puntuación separado (ej. `deepmultilingualpunctuation`).

In [ ]:
# Decodificamos la salida:
predicted_ids = torch.argmax(logits, dim=-1)   # seleccionamos las salidas (ids) de mayor probabilidad.
transcription = processor.decode(predicted_ids[0])   # obtenemos los tokens (palabras) con los ids.

In [ ]:
transcription   # observa que en este modelo la introducción y el cierre del audio se excluyen de manera automática.

'las ranas y el pantano seco vivían dos ranas en un bello pantano pero llegó el verano y se secó por lo cual lo abandonaron para buscar otro con agua aliaron en su camino un profundo poso repleto de agua y al verlo dijo una rana la otra amiga bajemos las dos a este poso pero is también se secara el agua deste pozo repuso la compañera cómo crees que subiremos entonces al tratar de emprender una acción analiza primro las consecuencias de ella fábula'

## 🌐 Caso 4 — Google Speech Recognition

El Caso 4 usa la librería `SpeechRecognition` de Python como interfaz al **servicio gratuito de reconocimiento de voz de Google** (el mismo motor que usa `google.com/voice`).

**Características clave:**
- No requiere API key ni cuenta de Google para uso básico.
- Gratuito pero con límites no documentados (aproximadamente 50 solicitudes/hora).
- Soporta ~125 idiomas y variantes regionales (ej. `es-MX`, `es-ES`, `es-AR`).
- **Requiere formato WAV** — no acepta MP3 directamente.

**Diferencia con los Casos anteriores:**

```
Casos 1, 3: Tu máquina ──────────────────────────────── transcripción
             (todo el procesamiento es local)

Caso 2:     Tu máquina ──── HTTPS ──► OpenAI API ──── transcripción
             (modelo en servidores de OpenAI, pago)

Caso 4:     Tu máquina ──── HTTPS ──► Google API ─── transcripción
             (modelo en servidores de Google, gratuito con límites)
```

**Cuándo usar el Caso 4:**
- Proyectos educativos o de prototipado donde el volumen es bajo.
- Cuando no quieres descargar modelos grandes localmente.
- Cuando necesitas soporte para un dialecto específico (ej. español mexicano `es-MX`).

# **Caso - 4: Google-Translator**

* #### **Esta opción está basada en el traductor de Google y no solo soporta una gran variedad de idiomas, sino también una buena variedad de acentos para un mismo idioma. En particular para el español existen versiones para diversos acentos de países de latinoamérica.**

* #### **Requiere que el archivo de entrada esté en formato de audio WAV.**

* #### **Es gratuita, pero tiene restricciones si hacemos muchas solicitudes en poco tiempo, en cuyo caso se requiere hacer uso de su API y por lo tanto tendrá un costo. Consulta la documentación.**

* #### **Puedes verifica los idiomas y variantes soportadas en la siguiente liga:**

https://cloud.google.com/speech-to-text/docs/speech-to-text-supported-languages?hl=es-419

* #### **Documentación de SpeechRecognition:** https://pypi.org/project/SpeechRecognition/

In [ ]:
!pip install SpeechRecognition pydub

In [ ]:
from pydub import AudioSegment
import speech_recognition as sr
import os

from IPython.display import Markdown, display

## 🔄 ¿Por qué convertir MP3 → WAV?

La librería `SpeechRecognition` solo soporta archivos WAV como entrada directa. La razón técnica es que WAV es PCM sin comprimir — `SpeechRecognition` accede directamente a las muestras de audio sin necesitar un decodificador de formato.

`pydub` realiza la conversión usando ffmpeg internamente:
```
20029-01.mp3  →  pydub.AudioSegment.from_mp3()  →  .export("tmp.wav", format="wav")
```

| Formato | Tipo | Tamaño (73 seg) | Requiere decodificación |
|---------|------|-----------------|------------------------|
| MP3 | Comprimido (lossy) | ~1.2 MB | Sí (ffmpeg) |
| WAV PCM | Sin comprimir | ~13 MB | No |

El archivo `tmp.wav` es temporal — se elimina al final con `os.remove(wav_file)` para no ocupar espacio innecesario.

📥 **Datos de entrada:** `audio_mp3_path` — ruta al archivo MP3 original.
📤 **Salida:** `wav_file` — archivo WAV temporal en disco.

In [ ]:
 # Se requiere formato WAV para usar esta solución de Google,
 # por lo que exportamos el mp3 a wav como sigue:

audio_mp3_path = "./20029-01.mp3"   # el audio de entrada.
wav_file = "./tmp.wav"      # el audio de salida.

audio = AudioSegment.from_mp3(audio_mp3_path)
audio.export(wav_file, format="wav")

<_io.BufferedRandom name='./tmp.wav'>

## 🎛️ Parámetros del reconocedor y manejo de errores

| Componente | Valor / Método | Descripción |
|-----------|---------------|-------------|
| `sr.Recognizer()` | Objeto | Inicializa el motor de reconocimiento. Un solo objeto puede reutilizarse para múltiples archivos |
| `sr.AudioFile(wav_file)` | Context manager | Abre el archivo WAV y lo prepara para lectura |
| `recognizer.record(source)` | Sin límite de tiempo | Lee todo el archivo. Con `duration=30` leerías solo los primeros 30 segundos |
| `recognizer.recognize_google()` | API call | Envía el audio a Google y devuelve el texto |
| `language="es-MX"` | Código BCP-47 | Variante de español mexicano. Otros: `"es-ES"`, `"es-AR"`, `"es-CO"` |

**Manejo de errores — dos excepciones posibles:**

| Excepción | Cuándo ocurre | Qué hacer |
|-----------|--------------|-----------|
| `sr.UnknownValueError` | Google no pudo entender el audio (silencio, ruido, idioma incorrecto) | Revisar calidad del audio o cambiar el idioma |
| `sr.RequestError` | Error de red o límite de tasa de la API de Google | Esperar y reintentar, o cambiar a otro caso |

**⚠️ Limitación importante:** `recognizer.record(source)` carga todo el archivo en memoria antes de enviarlo. Para archivos grandes (>10 MB descomprimidos ≈ >1 min de audio), `record()` puede ser lento. En producción dividir el archivo en chunks de 30 seg con `pydub`.

In [ ]:
# Inicializamos el reconocedor:
recognizer = sr.Recognizer()

# Cargamos y analizamos el archivo en formato WAV:
with sr.AudioFile(wav_file) as source:

    audio_data = recognizer.record(source)  # extracción de los datos del audio

    try:
        # Procedemos con la extracción del texto.
        # Estoy usando la pronunciación que tiene para México,
        # pero en dado caso puede cambiarse:
        texto = recognizer.recognize_google(audio_data,
                                            language="es-MX"
                                            )
        display(Markdown(texto))  # para darle cierto formato a la salida

    except sr.UnknownValueError:
        print("No se pudo entender el audio")
    except sr.RequestError as e:
        print(f"No se pudo solicitar resultados al servicio de reconocimiento; {e}")


las fábulas de Esopo grabado para librivox.org por Paulino fábula número 31 las ranas y el Pantano seco vivían dos ranas en un bello pantano Pero llegó el verano y se secó Por lo cual lo abandonaron para buscar otro con agua aliaron en su camino un profundo pozo repleto de agua y al verlo dijo una rana a la otra amiga bajemos las dos a este pozo pero también se secara el agua de este pozo repuso la compañera Cómo crees que subiremos entonces al tratar de emprender una acción analiza primero las consecuencias de ella fin de la fábula Esta es una grabación del dominio público

In [ ]:
texto

'las fábulas de Esopo grabado para librivox.org por Paulino fábula número 31 las ranas y el Pantano seco vivían dos ranas en un bello pantano Pero llegó el verano y se secó Por lo cual lo abandonaron para buscar otro con agua aliaron en su camino un profundo pozo repleto de agua y al verlo dijo una rana a la otra amiga bajemos las dos a este pozo pero también se secara el agua de este pozo repuso la compañera Cómo crees que subiremos entonces al tratar de emprender una acción analiza primero las consecuencias de ella fin de la fábula Esta es una grabación del dominio público'

In [ ]:
# removemos el archivo WAV temporal:
os.remove(wav_file)

## 📊 Comparación final — los 4 enfoques sobre el mismo audio

> **Nota:** El Caso 2 no fue ejecutado en este notebook (omitido por costo de API). Los datos de su columna provienen de la documentación oficial de OpenAI y del output conocido del modelo `whisper-1` sobre audio en español.

| Criterio | Caso 1 (Whisper HF) | Caso 2 (OpenAI API) ⚠️ no ejecutado | Caso 3 (Wav2Vec2) | Caso 4 (Google) |
|----------|---------------------|---------------------|-------------------|-----------------|
| **Puntuación en salida** | Mínima | ✅ Buena (post-proc. OpenAI) | ❌ Sin puntuación | Parcial |
| **Mayúsculas** | Correctas | Correctas | ❌ Todo minúsculas | Correctas |
| **Errores en la fábula** | Mínimos | Mínimos | "poso" por "pozo" | Mínimos |
| **Timestamps** | ✅ Sí | Solo con `verbose_json` | ❌ No | ❌ No |
| **API key requerida** | No | ✅ Sí (OpenAI, pago) | No (HF login) | No |
| **Datos enviados a la nube** | No | ✅ Sí (OpenAI) | No | ✅ Sí (Google) |
| **Formato de entrada** | MP3/WAV/M4A | MP3/WAV/M4A | MP3 (re-muestreado) | **WAV solo** |
| **GPU requerida** | Recomendada | No | Recomendada | No |
| **Costo operativo** | Infraestructura | ~$0.006/min de audio | Infraestructura | Gratis (con límites) |
| **Ejecutado en este NB** | ✅ Sí | ⚠️ Omitido (costo) | ✅ Sí | ✅ Sí |

**Recomendación según caso de uso:**
```
Datos sensibles + calidad alta        → Caso 1 (Whisper local)
Alto volumen + integración rápida     → Caso 2 (OpenAI API)
Completamente local + sin API key     → Caso 3 (Wav2Vec2)
Prototipado rápido + bajo volumen     → Caso 4 (Google)
Generación de subtítulos (SRT/VTT)    → Caso 1 (timestamps) o Caso 2 con verbose_json
```

**¿Por qué Caso 1 y Caso 2 producen output similar?**
Ambos usan el mismo modelo Whisper de OpenAI. La diferencia real es el **entorno de ejecución**:
- Caso 1: tu hardware, sin costo por llamada, sin envío de datos a terceros.
- Caso 2: servidores OpenAI, costo por minuto de audio, mejor puntuación por post-procesamiento adicional.